# Covariates fit

Which covariates should drive the transitions of the AR(1)-HMM(b,1) (`drivers/models/ar1_hmm_beta_1.py`)?

Each covariate set in `COVARIATE_SETS` is fitted the same way the drivers do it:
1. `HMM(b,1)` is seeded from the saved `hmm_1` fit and fitted with that set (ridge λ_β = 1e-2).
2. `AR(1)-HMM(b,1)` takes its emission from the saved `ar1_hmm_1` and its transition from step 1, then it is refitted.

The fits are compared out-of-sample on the validation split, using rolling k-step forecasts (RMSE / R² / MAPE). Train NLL, AIC and BIC are reported as secondary information.
Fits are cached under `results/{room}/{tag}/states_{K}/covariates/`, so the driver pickles are never touched.

In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import os
import re

import equinox as eqx
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from drivers.models import hmm_beta_1
from drivers.models.evaluate import metrics_table, rolling_predictions
from drivers.models.utils import (COVARIATE_COLS, HORIZONS, OBS_COLS, PLOT_HORIZONS, RESULTS_DIR, Config, Loss,
                                  fit_and_report, load_data, load_fit, negative_likelihood, result_path)
from drivers.utils import load_model, save_model
from src.api.v5 import ForwardAlgorithm, Params

## Configuration

Edit this cell to choose the covariate sets. Valid names are `off_day`, `working_hours`, and `sin_/cos_{tod,week}_{k}` for any harmonic `k`.
The empty set `()` is the static `ar1_hmm_1` baseline. It is loaded from disk, not refitted.
Each new set costs two L-BFGS fits plus a JAX compile, so expect a few minutes per set.

In [2]:
ROOM, K, TAG = "room_015", 6, "30min"
LOSS = Loss(beta_lam=1e-2)   # same ridge on beta as the ar1_hmm_beta_1 driver
REFIT = False                # True ignores the cached fits
DEFAULT_COLS = ("off_day", "sin_week_1", "cos_week_1", "sin_tod_1", "cos_tod_1", "working_hours")  # the drivers' old default

COVARIATE_SETS = {
    "none (ar1_hmm_1)": (),
    "default": DEFAULT_COLS,
    "off_day": ("off_day",),
    "working_hours": ("working_hours",),
    "off_day+working_hours": ("off_day", "working_hours"),
    #"tod_1": ("sin_tod_1", "cos_tod_1"),
    #"tod_1_2": ("sin_tod_1", "cos_tod_1", "sin_tod_2", "cos_tod_2"),
    #"off_day+tod_1": ("off_day", "sin_tod_1", "cos_tod_1"),
    #"default+tod_2": (*DEFAULT_COLS, "sin_tod_2", "cos_tod_2"),
}

CACHE_DIR = os.path.join(RESULTS_DIR, ROOM, TAG, f"states_{K}", "covariates")

## Fitting

In [3]:
def cache_path(name: str) -> str:
    return os.path.join(CACHE_DIR, re.sub(r"[^A-Za-z0-9_+-]+", "_", name).strip("_") + ".pkl")


def fit_set(name: str, cols: tuple[str, ...]) -> dict:
    """Fit HMM(b,1) then AR(1)-HMM(b,1) with covariates `cols`, or load the cached fit."""
    cfg = Config(ROOM, OBS_COLS, cols, K, loss=LOSS if cols else Loss(), tag=TAG)
    if not cols:  # static baseline
        return {"params": load_model(result_path(cfg, "ar1_hmm_1"))["result"].params, "cfg": cfg}

    path = cache_path(name)
    if os.path.exists(path) and not REFIT:
        saved = load_model(path)
        if tuple(saved["covariate_cols"]) == tuple(cols):
            print(f"Loaded {name} from {path}")
            return {"params": saved["result"].params, "cfg": cfg}

    train = load_data(cfg)
    hmm_beta = fit_and_report(f"HMM(b,1) [{name}]", hmm_beta_1.build(cfg, train), train.dynamic(), cfg)
    params = Params(emission=load_fit(cfg, "ar1_hmm_1").emission, transition=hmm_beta.params.transition)
    result = fit_and_report(f"AR(1)-HMM(b,1) [{name}]", params, train.dynamic(), cfg)
    save_model({"result": result, "config": cfg, "covariate_cols": list(cols), "hmm_beta": hmm_beta}, path)
    return {"params": result.params, "cfg": cfg}


fits = {name: fit_set(name, cols) for name, cols in COVARIATE_SETS.items()}

Loaded default from /Users/madshaakonsson/Desktop/7-semester/hmm-modelling/results/room_015/30min/states_6/covariates/default.pkl
Loaded off_day from /Users/madshaakonsson/Desktop/7-semester/hmm-modelling/results/room_015/30min/states_6/covariates/off_day.pkl
--- Fitting HMM(b,1) [working_hours] (room_015, obs=['co2'], K=6, loss=ridge_beta_lam0.01)
Final loss: 14363.543905477172
Negative log-likelihood (unpenalised): 14361.590815235899
Converged: False
Number of clipped steps in f_t: 0
--- Fitting AR(1)-HMM(b,1) [working_hours] (room_015, obs=['co2'], K=6, loss=ridge_beta_lam0.01)
Final loss: 12262.714241451808
Negative log-likelihood (unpenalised): 12261.287982034679
Converged: False
Number of clipped steps in f_t: 0
--- Fitting HMM(b,1) [off_day+working_hours] (room_015, obs=['co2'], K=6, loss=ridge_beta_lam0.01)
Final loss: 14358.992507744777
Negative log-likelihood (unpenalised): 14356.581056594674
Converged: False
Number of clipped steps in f_t: 0
--- Fitting AR(1)-HMM(b,1) [off_d

## Evaluation

These are the rolling k-step forecasts of the validation split (filter up to t, predict t+k), the same metrics the drivers write to `predictive_metrics.csv`.
`val_nll` is the one-step NLL of the validation part of one filter pass over train and validation.

In [4]:
def n_params(params: Params) -> int:
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(eqx.filter(params, eqx.is_inexact_array)))


def covariates_or_none(params: Params, data):
    return data.xs if hasattr(params.transition, "beta") else None


def val_nll(params: Params, train, val) -> float:
    xs, xs_val = covariates_or_none(params, train), covariates_or_none(params, val)
    ys = jnp.concatenate([train.ys, val.ys])
    xs = None if xs is None else jnp.concatenate([xs, xs_val])
    _, f_t, _ = ForwardAlgorithm().run(params=params, ys=ys, xs=xs, initial_dist=train.init_dist)
    return float(-jnp.sum(jnp.log(f_t[len(train.ys):])))


metrics, summary, preds = [], [], {}
for name, fit in fits.items():
    params, cfg = fit["params"], fit["cfg"]
    train, val = load_data(cfg), load_data(cfg, "val")
    preds[name] = rolling_predictions(params, train, val)
    metrics.append(metrics_table(name, "ar1_hmm_beta_1", cfg, np.asarray(val.ys)[:, None], preds[name]))

    xs = covariates_or_none(params, train)
    nll = float(negative_likelihood(params, (train.ys, xs, train.init_dist)))
    p, n = n_params(params), len(train.ys)
    summary.append({"set": name, "n_cov": len(cfg.covariate_cols), "n_params": p, "train_nll": nll,
                    "val_nll": val_nll(params, train, val), "AIC": 2 * nll + 2 * p, "BIC": 2 * nll + p * np.log(n)})

metrics = pd.concat(metrics, ignore_index=True).rename(columns={"model": "set"})
val_data = load_data(next(iter(fits.values()))["cfg"], "val")

## Summary

In [5]:
wide = metrics.pivot(index="set", columns="horizon", values=["RMSE", "R2", "MAPE"])
wide.columns = [f"{m}_h{h}" for m, h in wide.columns]
table = pd.DataFrame(summary).set_index("set").join(wide).sort_values(f"RMSE_h{PLOT_HORIZONS[0]}")

lower_better = [c for c in table.columns if c.startswith(("RMSE", "MAPE")) or c in ("train_nll", "val_nll", "AIC", "BIC")]
higher_better = [c for c in table.columns if c.startswith("R2")]
try:  # Styler needs jinja2
    shown_table = (table.style.format(precision=3)
                   .highlight_min(subset=lower_better, color="#c6efce")
                   .highlight_max(subset=higher_better, color="#c6efce"))
except AttributeError:
    shown_table = table.round(3)
shown_table

,n_cov,n_params,train_nll,val_nll,AIC,BIC,RMSE_h1,RMSE_h2,RMSE_h4,RMSE_h6,RMSE_h8,R2_h1,R2_h2,R2_h4,R2_h6,R2_h8
set,,,,,,,,,,,,,,,,
off_day+working_hours,2,108,12250.510,2900.617,24717.019,25346.361,167.118,239.445,307.471,337.167,368.179,0.898,0.790,0.653,0.583,0.503
working_hours,1,78,12261.288,2905.391,24678.576,25133.101,167.496,240.109,308.686,337.705,366.967,0.897,0.788,0.650,0.582,0.506
default,6,228,12158.919,2903.795,24773.838,26102.449,167.273,239.468,310.825,342.656,376.888,0.897,0.790,0.646,0.569,0.479
off_day,1,78,12315.612,2938.301,24787.223,25241.748,167.438,241.609,320.916,372.004,428.948,0.897,0.786,0.622,0.492,0.325
none (ar1_hmm_1),0,48,12331.493,2941.807,24758.985,25038.693,167.417,241.580,321.115,372.813,430.879,0.897,0.786,0.622,0.490,0.319


## Likelihood-ratio test: `default` vs `off_day+working_hours`

`off_day+working_hours` is nested in `default`: it is `default` with the β of `sin_week_1`, `cos_week_1`, `sin_tod_1` and `cos_tod_1` set to 0.
Under H₀ (those β are all 0),

$$\Lambda = 2\,(\ell_{\text{full}} - \ell_{\text{restricted}}) \;\overset{a}{\sim}\; \chi^2_{df}, \qquad df = 4 \cdot K(K-1),$$

since every covariate has one β per row and per non-reference column of Γ.

Caveats:
- ℓ is the **unpenalised** train log-likelihood evaluated at the ridge (λ_β = 1e-2) estimates, so it is slightly below the true maximum of each model. The penalty is tiny compared with ℓ, so the effect on Λ is small.
- The χ² approximation assumes the fits reached their optima, and the solver reported `Converged: False`. A negative Λ (restricted fits better) means an optimisation problem, not evidence for H₀.

In [ ]:
from scipy.stats import chi2


def lrt(full: str, restricted: str) -> pd.DataFrame:
    # Λ = 2(ℓ_full − ℓ_restricted) ~ χ²(df) under H0: the β of the covariates only in `full` are 0
    full_cols, restricted_cols = set(COVARIATE_SETS[full]), set(COVARIATE_SETS[restricted])
    if not restricted_cols <= full_cols:
        raise ValueError(f"'{restricted}' is not nested in '{full}'")
    lr = 2 * (table.loc[restricted, "train_nll"] - table.loc[full, "train_nll"])
    df = int(table.loc[full, "n_params"] - table.loc[restricted, "n_params"])
    return pd.DataFrame([{
        "full": full, "restricted": restricted, "H0: β = 0 for": ", ".join(sorted(full_cols - restricted_cols)),
        "ll_full": -table.loc[full, "train_nll"], "ll_restricted": -table.loc[restricted, "train_nll"],
        "LR": lr, "df": df, "chi2_crit_5%": chi2.ppf(0.95, df), "p_value": chi2.sf(lr, df) if lr > 0 else 1.0,
    }]).set_index(["full", "restricted"])


lrt_result = lrt("default", "off_day+working_hours")
lrt_result

## Predictive performance across horizons

In [6]:
order = list(table.index)
colors = dict(zip(order, plt.cm.tab10(np.linspace(0, 1, 10))[:len(order)])) if len(order) <= 10 else \
         dict(zip(order, plt.cm.tab20(np.linspace(0, 1, len(order)))))

fig, (ax_rmse, ax_r2, ax_mape) = plt.subplots(1, 3, figsize=(18, 5))
for name in order:
    m = metrics[metrics["set"] == name].sort_values("horizon")
    style = dict(color=colors[name], marker="o", lw=2.5 if name == order[0] else 1.2)
    ax_rmse.plot(m["horizon"], m["RMSE"], label=name, **style)
    ax_r2.plot(m["horizon"], m["R2"], label=name, **style)
    ax_mape.plot(m["horizon"], m["MAPE"], label=name, **style)
ax_rmse.set(title="Validation RMSE", xlabel="Horizon k (steps of 30 min)", ylabel="RMSE [ppm]", xticks=HORIZONS)
ax_r2.set(title="Validation R²", xlabel="Horizon k (steps of 30 min)", ylabel="R²", xticks=HORIZONS)
ax_mape.set(title="Validation MAPE", xlabel="Horizon k (steps of 30 min)", ylabel="MAPE [%]", xticks=HORIZONS)
ax_r2.legend(loc="lower left", fontsize=8)
fig.suptitle(f"AR(1)-HMM(b,1) covariate sets — {ROOM}, K={K}")
fig.tight_layout()

In [10]:
fig, ax = plt.subplots(figsize=(12, 4.5))
width = 0.8 / len(PLOT_HORIZONS)
x = np.arange(len(order))
for i, k in enumerate(PLOT_HORIZONS):
    ax.bar(x + i * width, table.loc[order, f"RMSE_h{k}"], width, label=f"k={k}")
ax.set_xticks(x + width * (len(PLOT_HORIZONS) - 1) / 2, order, rotation=30, ha="right")
ax.set(ylabel="RMSE [ppm]", title=f"Validation RMSE at long horizons (sorted by k={PLOT_HORIZONS[0]})")
ax.legend()
fig.tight_layout()

Here is the change in RMSE relative to the `default` covariate set. Negative (blue) means the set forecasts better than the default.

In [11]:
ref = "default" if "default" in table.index else order[0]
rmse = table[[f"RMSE_h{k}" for k in HORIZONS]]
delta = rmse - rmse.loc[ref]

fig, ax = plt.subplots(figsize=(8, 0.5 * len(delta) + 1.5))
lim = np.nanmax(np.abs(delta.to_numpy())) or 1
im = ax.imshow(delta.to_numpy(), cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(HORIZONS)), [f"k={k}" for k in HORIZONS])
ax.set_yticks(range(len(delta)), delta.index)
for i in range(delta.shape[0]):
    for j in range(delta.shape[1]):
        ax.text(j, i, f"{delta.iat[i, j]:+.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label=f"RMSE − RMSE[{ref}] [ppm]")
ax.set_title(f"ΔRMSE vs '{ref}'")
fig.tight_layout()

## Forecasts over the first validation week

In [12]:
k = PLOT_HORIZONS[0]
shown = list(dict.fromkeys([*order[:3], "default", "none (ar1_hmm_1)"]))
shown = [s for s in shown if s in preds]

dt = val_data.datetime
week = np.asarray(dt < dt[0] + pd.Timedelta(days=7))
y_val = np.asarray(val_data.ys)

fig, axes = plt.subplots(len(shown), 1, figsize=(14, 2.6 * len(shown)), sharex=True, squeeze=False)
for ax, name in zip(axes[:, 0], shown):
    ax.plot(dt[week], y_val[week], color="black", lw=1.2, label="Observed")
    ax.plot(dt[week], preds[name][k][week, 0], color=colors[name], lw=1.2, label=f"{name} (k={k})")
    ax.set_title(f"{name}   RMSE={table.loc[name, f'RMSE_h{k}']:.1f}   R²={table.loc[name, f'R2_h{k}']:.3f}", loc="left")
    ax.set_ylabel("CO₂ [ppm]")
    ax.legend(loc="upper right", fontsize=8)
fig.suptitle(f"{k}-step rolling forecasts, first validation week (metrics over the full validation set)")
fig.tight_layout()

## Conclusion

Best set by validation RMSE at k=6: _fill in after running_.